# Análisis de Estrategias de Fusión - HybridRank RAG

Este notebook analiza los resultados de la evaluación comparativa
de diferentes estrategias de fusión para el sistema HybridRank RAG.

**Prerequisito**: Ejecutar `experiments/evaluate_fusion_strategies.py` para generar los datos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

RESULTS_DIR = Path('../experiments/results')
df = pd.read_csv(RESULTS_DIR / 'fusion_metrics.csv')
print(f'Total filas: {len(df)}')
print(f'Estrategias: {df["strategy"].nunique()}')
print(f'Queries: {df["query_id"].nunique()}')
df.head()

## 1. Vista General: Tabla Agregada por Estrategia

In [ ]:
# Identificar columnas de métricas
metric_cols = [c for c in df.columns if any(
    m in c for m in ['recall', 'precision', 'f1', 'mrr', 'map', 'ndcg']
)]

# Resumen por estrategia (promedio de todas las queries y k_values)
summary = df.groupby('strategy')[metric_cols].mean().round(4)
summary = summary.sort_values(metric_cols[0], ascending=False)

# Mostrar con estilo
summary.style.background_gradient(cmap='RdYlGn', axis=0)

## 2. Comparación Baseline vs Fusión por Métrica

In [ ]:
# Separar baselines de estrategias de fusión
baselines = ['bm25_only', 'dense_only']
fusion_strategies = [s for s in df['strategy'].unique() if s not in baselines]

# Métricas a comparar (para k=10)
df_k10 = df[df['top_k'] == 10].copy()
summary_k10 = df_k10.groupby('strategy')[metric_cols].mean().round(4)

# Gráfico de barras comparativo
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(metric_cols[:6]):
    ax = axes[i]
    data = summary_k10[col].sort_values(ascending=True)
    colors = ['#2196F3' if name in baselines else '#4CAF50' 
              if 'hybridrank' in name else '#FFC107'
              for name in data.index]
    data.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(col.upper(), fontweight='bold')
    ax.set_xlabel('Score')

plt.tight_layout()
plt.suptitle('Comparación de Estrategias (k=10)', y=1.02, fontsize=14, fontweight='bold')
plt.savefig(RESULTS_DIR / 'comparison_k10.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Análisis por Tipo de Query

In [ ]:
# Análisis por query_type (literal vs semántica)
if 'query_type' in df.columns and df['query_type'].nunique() > 1:
    pivot = df.pivot_table(
        index='strategy',
        columns='query_type',
        values=metric_cols[:3],
        aggfunc='mean'
    ).round(4)
    
    print('Rendimiento por tipo de query:')
    display(pivot)
    
    # Gráfico para cada tipo de query
    query_types = df['query_type'].unique()
    fig, axes = plt.subplots(1, len(query_types), figsize=(14, 6))
    if len(query_types) == 1:
        axes = [axes]
    
    for ax, qt in zip(axes, query_types):
        df_qt = df[(df['query_type'] == qt) & (df['top_k'] == 10)]
        data = df_qt.groupby('strategy')[metric_cols[0]].mean().sort_values()
        data.plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f'Query Type: {qt}', fontweight='bold')
        ax.set_xlabel(metric_cols[0])
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'by_query_type.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Solo hay un tipo de query o falta la columna query_type')

## 4. Efecto de los Parámetros en HybridRank

In [ ]:
# Filtrar solo estrategias HybridRank
hr_strategies = [s for s in df['strategy'].unique() if 'hybridrank' in s]

if hr_strategies:
    df_hr = df[df['strategy'].isin(hr_strategies) & (df['top_k'] == 10)]
    hr_summary = df_hr.groupby('strategy')[metric_cols].mean().round(4)
    
    print('Configuraciones HybridRank (k=10):')
    display(hr_summary)
    
    # Visualizar como heatmap
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(hr_summary.values, cmap='YlGn', aspect='auto')
    
    ax.set_xticks(range(len(hr_summary.columns)))
    ax.set_xticklabels(hr_summary.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(hr_summary.index)))
    ax.set_yticklabels(hr_summary.index)
    
    # Agregar valores en celdas
    for i in range(len(hr_summary.index)):
        for j in range(len(hr_summary.columns)):
            text = ax.text(j, i, f'{hr_summary.values[i, j]:.3f}',
                          ha='center', va='center', fontsize=9)
    
    plt.colorbar(im)
    plt.title('HybridRank: Rendimiento por Configuración', fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'hybridrank_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No se encontraron estrategias HybridRank en los resultados')

## 5. Mejora sobre Baselines

In [ ]:
# Calcular mejora porcentual sobre el mejor baseline
df_k10 = df[df['top_k'] == 10].copy()
strategy_means = df_k10.groupby('strategy')[metric_cols].mean()

# Mejor baseline por métrica
baseline_means = strategy_means.loc[strategy_means.index.isin(baselines)]
best_baseline = baseline_means.max()

# Mejora porcentual
improvement = ((strategy_means - best_baseline) / best_baseline * 100).round(2)
improvement = improvement.loc[~improvement.index.isin(baselines)]

print('Mejora porcentual sobre mejor baseline (k=10):')
print('(Valores positivos = supera al mejor baseline)\n')
display(improvement.style.applymap(
    lambda v: 'color: green; font-weight: bold' if v > 0 else 'color: red'
))

## 6. Conclusiones

In [ ]:
# Mejor estrategia general
df_k10 = df[df['top_k'] == 10]
overall_means = df_k10.groupby('strategy')[metric_cols].mean()

print('=== MEJORES ESTRATEGIAS POR MÉTRICA (k=10) ===\n')
for col in metric_cols:
    best = overall_means[col].idxmax()
    score = overall_means[col].max()
    print(f'  {col:20s}: {best:30s} ({score:.4f})')

# Ranking general (promedio de todas las métricas normalizadas)
normalized = (overall_means - overall_means.min()) / (overall_means.max() - overall_means.min())
overall_rank = normalized.mean(axis=1).sort_values(ascending=False)

print('\n=== RANKING GENERAL (score normalizado promedio) ===\n')
for i, (strategy, score) in enumerate(overall_rank.items(), 1):
    marker = '★' if 'hybridrank' in strategy else '  '
    print(f'  {i:2d}. {marker} {strategy:30s} {score:.4f}')